# Estudo toy: cobertura PEF (dump ↔ lake) + diagnóstico de match baixo

**Resumo.** Probe de baixo custo da **cobertura descritiva** entre o dump de pesquisa da Inteligência Fiscal (`ajuizamento.PEF`) e a tabela silver FACE do TJSP, em **janela intermediária** de ajuizamento (2017–2020). Nesta versão: (1) diagnóstico dos universos A/B (formatos CNJ); (2) **escada de match** exact → digits-only → máscara CNJ / `num_processo_limpo`; (3) tops estratificados; (4) grafos bipartidos exploratórios (tops only), com figuras Plotly interativas **e** export estático (PNG/SVG via Kaleido). **Não** é auditoria estadual, sobrevivência, nem score “contumaz”.

**Escopo.** Oversample + filtro local de datas; join lake com ≤250 PEFs distintos (amostra estratificada por ano) via DuckDB `WHERE … IN (…)`. Sem scan completo do lake. Sem PII.

**Freeze.** Dump `extracao` do `.env` (tipicamente `2026-03`). Lake: `SILVER_FACE_CLEAN` via `config.paths`.

**Chaves FACE (schema).** Join PEF ↔ lake usa `numero` (CNJ mascarado, ~25 caracteres) e, na normalização, a forma de 20 dígitos (equivalente conceitual a `num_processo_limpo`). `cd_processo` é identificador interno E-SAJ (alfanumérico, ~13 chars) — **nunca** substituto do PEF. `controle` é número local de foro (`YYYY/NNNNNN`), também **não** CNJ.


## 1. Conceitos e termos

### PEF
**PEF** (*Processo de Execução Fiscal*) = número CNJ do processo no dump (`ajuizamento.PEF`). Identifica a execução fiscal ajuizada para um crédito inscrito (`ID_DEBITO`).

### Identificadores FACE (não confundir)
| Identificador | Forma típica | Papel no join |
|---|---|---|
| Dump `PEF` | CNJ (máscara ou dígitos) | chave esquerda |
| FACE `numero` | CNJ mascarado (~25 chars) | chave direita nominal |
| FACE `num_processo_limpo` | 20 dígitos | chave direita normalizada (digits-only) |
| FACE `cd_processo` | E-SAJ ~13 alfanumérico | **não** usar como PEF |
| FACE `controle` | `YYYY/NNNNNN` (foro-local) | **não** CNJ |

Assumir `PEF == cd_processo` produz cobertura zero silenciosamente.

### Cobertura descritiva (definição operacional)
Neste notebook medimos **cobertura descritiva** (match rate), não linkage probabilístico à Fellegi–Sunter (1969). Seja $N$ o tamanho do *join set* e $M$ o número de PEFs com ao menos uma chave alinhada em FACE:

$$
\hat{c} = \frac{M}{N} = \frac{1}{N}\sum_{i=1}^{N}\mathbf{1}\{\exists\, j:\; K(i)=K'(j)\},
$$

onde $K$ / $K'$ são transformações de chave (identidade, digits-only, máscara CNJ). Reportamos $\hat{c}$ por degrau da escada; **não** estimamos pesos $m$/$u$ nem posterior de match.

### Dump vs lake (dois universos)
| Camada | Papel | Chave neste toy |
|---|---|---|
| Dump (API / parquet local) | Metadados administrativos de crédito e ajuizamento | `ajuizamento.PEF` |
| Lake FACE silver | Metadados processuais TJSP (scraped) | `numero` / forma 20 dígitos |

Cobertura deve ser medida **antes** de features processuais ou modelos de duração.

### Janela temporal intermediária
`extracao=2026-03` congela o dump, mas filings muito recentes são “cedo demais” para maturidade processual. Mantemos **2017-01-01 … 2020-12-31** salvo indicação contrária nos diagnósticos abaixo.

### Framing teórico (analogia apenas)
Modelos de compliance (Allingham & Sandmo, 1972; Andreoni et al., 1998) e detecção não supervisionada em dados tributários (Savić et al., 2022) motivam o *interesse* em litígio fiscal, mas **não** são evidência empírica de execução fiscal SP + ML. Este toy é diagnóstico de join.


## 2. Setup (caminhos, limites, segurança)

Carrega o mono, `.env`, cliente da API e caminhos do lake. Limites duros para timeout e memória. Janela `WINDOW_*` é o filtro aplicado **antes** do join.

Figuras: Plotly (`fig.show()`) + export estático PNG/SVG em `playground/output/pef_toy_figures/` via Kaleido (`fig.write_image`).


In [1]:
from __future__ import annotations

import os
import re
import sys
from pathlib import Path

import duckdb
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from dotenv import load_dotenv

NB_DIR = Path.cwd()
MONO = next((p for p in [NB_DIR, *NB_DIR.parents] if (p / "shared" / "cemepi_api").exists()), NB_DIR)
sys.path.insert(0, str(MONO))
sys.path.insert(0, str(MONO / "pipelines" / "litigancia" / "src"))
sys.path.insert(0, str(MONO / "shared"))
os.chdir(MONO)
load_dotenv(MONO / ".env")

from config.paths import REPO_ROOT, SILVER_FACE_CLEAN, LAKE_ROOT
from cemepi_api.client import CemepiClient, load_settings

# Oversample then filter locally (API may lack date filter)
AJUIZ_OVERSAMPLE = 2000
DEBITO_LIMIT = 300
PEF_JOIN_CAP = 250  # DuckDB IN <=300
DUCKDB_MEM = "2GB"
FACE_FORMAT_SAMPLE = 500  # LIMIT-only diagnostic, not a join key list
RANDOM_STATE = 42

# Mid-window: maturity vs recent tails (documented; unchanged unless diag says otherwise)
WINDOW_START = pd.Timestamp("2017-01-01")
WINDOW_END = pd.Timestamp("2020-12-31")

assert REPO_ROOT == MONO
assert SILVER_FACE_CLEAN.exists(), f"missing FACE silver: {SILVER_FACE_CLEAN}"

S = load_settings()
C = CemepiClient(S)
SAMPLE_DIR = C.sample_dir()
DUMP_ROOT = S.dump_root

FIG_DIR = (
    MONO / "projects" / "litigancia" / "notebooks" / "playground" / "output" / "pef_toy_figures"
)
FIG_DIR.mkdir(parents=True, exist_ok=True)


def show_and_save(fig, stem: str, *, width: int = 960, height: int | None = None) -> Path:
    """Interactive show + static PNG/SVG via Kaleido (publication-ready)."""
    if height is not None:
        fig.update_layout(height=height)
    fig.update_layout(width=width, template="plotly_white", font=dict(size=12))
    fig.show()
    png = FIG_DIR / f"{stem}.png"
    svg = FIG_DIR / f"{stem}.svg"
    kw = dict(width=width, scale=2)
    if height is not None:
        kw["height"] = height
    fig.write_image(str(png), **kw)
    fig.write_image(str(svg), **{k: v for k, v in kw.items() if k != "scale"})
    print(f"static → {png.name}, {svg.name}")
    return png


print("REPO_ROOT", REPO_ROOT)
print("extracao", f"{S.ano:04d}-{S.mes:02d}")
print("DUMP_ROOT exists", DUMP_ROOT.exists())
print("LAKE_ROOT exists", LAKE_ROOT.exists())
print("SILVER_FACE_CLEAN", SILVER_FACE_CLEAN.name)
print("FIG_DIR", FIG_DIR)
print("window", WINDOW_START.date(), "…", WINDOW_END.date())
print("limits", {"oversample": AJUIZ_OVERSAMPLE, "debito": DEBITO_LIMIT, "pef_join": PEF_JOIN_CAP})


REPO_ROOT /Users/etorebraga/Code/cemepi-ctf-intel-fiscal
extracao 2026-03
DUMP_ROOT exists True
LAKE_ROOT exists True
SILVER_FACE_CLEAN face_processos_clean_delta
FIG_DIR /Users/etorebraga/Code/cemepi-ctf-intel-fiscal/projects/litigancia/notebooks/playground/output/pef_toy_figures
window 2017-01-01 … 2020-12-31
limits {'oversample': 2000, 'debito': 300, 'pef_join': 250}


## 3. Probe de datas de ajuizamento (antes do filtro)

Objetivo: estimar a distribuição empírica de `DT_AJUIZAMENTO` em uma amostra maior (API oversample e/ou parquets locais sob `$DUMP_ROOT/.../samples`), reportando mínimo, mediana e máximo. A janela intermediária $W=[t_0,t_1]$ é escolhida **depois** deste probe:

$$
t_{\mathrm{med}} = \mathrm{median}\{DT_i\}, \qquad W = [2017\text{-}01\text{-}01,\, 2020\text{-}12\text{-}31]
$$

(centrada na maturidade processual vs. caudas recentes 2024+). Sem figura nesta seção — apenas estatísticas descritivas.


In [2]:
def _normalize_aj(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    need = ["ID_DEBITO", "PEF", "DT_AJUIZAMENTO", "NOMECOMARCA"]
    for c in need:
        if c not in out.columns:
            out[c] = pd.NA
    out = out[need]
    out["PEF"] = out["PEF"].astype(str).str.strip().replace({"nan": pd.NA, "None": pd.NA})
    out["DT_AJUIZAMENTO"] = pd.to_datetime(out["DT_AJUIZAMENTO"], errors="coerce")
    return out


frames: list[pd.DataFrame] = []
api_ok = False
api_err = None
try:
    aj_api = C.sample_dataset(
        "ajuizamento",
        limit=AJUIZ_OVERSAMPLE,
        select="ID_DEBITO,PEF,DT_AJUIZAMENTO,NOMECOMARCA",
    )
    frames.append(_normalize_aj(aj_api).assign(_src="api"))
    api_ok = True
except Exception as e:  # noqa: BLE001 — timeout / API flake → dump fallback
    api_err = repr(e)
    print("API oversample failed; will rely on local dump samples:", api_err)

local_dir = DUMP_ROOT / f"extracao={S.ano:04d}-{S.mes:02d}" / "samples"
local_paths = sorted(local_dir.glob("*ajuizamento*.parquet")) if local_dir.exists() else []
for p in local_paths:
    try:
        frames.append(_normalize_aj(pd.read_parquet(p)).assign(_src=f"dump:{p.name}"))
    except Exception as e:  # noqa: BLE001
        print("skip local", p.name, e)

assert frames, "no ajuizamento frames from API or DUMP_ROOT samples"
aj_probe = pd.concat(frames, ignore_index=True)
aj_probe = aj_probe.dropna(subset=["PEF"]).drop_duplicates(subset=["PEF"], keep="first")

d = aj_probe["DT_AJUIZAMENTO"]
probe_stats = {
    "n_unique_pef": int(aj_probe["PEF"].nunique()),
    "n_with_date": int(d.notna().sum()),
    "min": str(d.min()) if d.notna().any() else None,
    "median": str(d.median()) if d.notna().any() else None,
    "max": str(d.max()) if d.notna().any() else None,
    "api_ok": api_ok,
    "n_local_files": len(local_paths),
}
year_counts_all = d.dt.year.value_counts().sort_index()
print("probe_stats", probe_stats)
print("year counts (unique PEF, all dates):")
print(year_counts_all.to_string())
print(
    f"\nChosen window: {WINDOW_START.date()} … {WINDOW_END.date()} "
    "(mid band: maturity of cases vs very recent filings; excludes 2024–2026-recent tails)."
)


probe_stats {'n_unique_pef': 1201, 'n_with_date': 1201, 'min': '2016-01-13 00:00:00', 'median': '2018-11-26 00:00:00', 'max': '2024-01-18 00:00:00', 'api_ok': True, 'n_local_files': 4}
year counts (unique PEF, all dates):
DT_AJUIZAMENTO
2016    417
2017    143
2018    409
2019     73
2020    143
2021     12
2022      3
2024      1

Chosen window: 2017-01-01 … 2020-12-31 (mid band: maturity of cases vs very recent filings; excludes 2024–2026-recent tails).


## 4. Filtro da janela intermediária (antes do join)

Aplicamos o predicado $t_0 \le DT\_AJUIZAMENTO \le t_1$ **antes** de qualquer métrica de cobertura. Persistimos a amostra filtrada em parquet local sob `SAMPLE_DIR` (não no repo habitual-tax). Toda taxa $\hat{c}$ abaixo é condicional a essa janela.


In [3]:
aj = aj_probe.loc[
    aj_probe["DT_AJUIZAMENTO"].between(WINDOW_START, WINDOW_END, inclusive="both")
].copy()
aj_path = SAMPLE_DIR / "toy_ajuizamento_midwindow.parquet"
aj.to_parquet(aj_path, index=False)

print("mid-window rows", len(aj), "| unique PEF", aj["PEF"].nunique(), "| unique ID_DEBITO", aj["ID_DEBITO"].nunique())
print("mid-window year counts:")
print(aj["DT_AJUIZAMENTO"].dt.year.value_counts().sort_index().to_string())
print("saved", aj_path)

deb = None
deb_err = None
try:
    deb = C.sample_dataset(
        "debito",
        limit=DEBITO_LIMIT,
        select=(
            "ID_DEBITO,TIPO_DEBITO,SITUACAO_DEBITO,"
            "STATUS_AJUIZAMENTO_DEBITO,ANO_EXTRACAO,MES_EXTRACAO"
        ),
        extra={"ANO_EXTRACAO": S.ano, "MES_EXTRACAO": S.mes},
    )
    deb_path = SAMPLE_DIR / "toy_debito_sample_v2.parquet"
    deb.to_parquet(deb_path, index=False)
    print("debito rows", len(deb), "| unique ID_DEBITO", deb["ID_DEBITO"].nunique(), "| saved", deb_path.name)
except Exception as e:  # noqa: BLE001
    deb_err = repr(e)
    print("debito sample failed:", deb_err)

if deb is not None:
    merged = aj.merge(deb, on="ID_DEBITO", how="left", suffixes=("", "_deb"))
    n_hit = int(merged["TIPO_DEBITO"].notna().sum()) if "TIPO_DEBITO" in merged.columns else 0
    print("ID_DEBITO overlap (mid-window aj × debito sample):", n_hit)
    if n_hit == 0:
        print("Note: independent API heads often yield 0 overlap; full dump join is future work.")
else:
    merged = aj.copy()

display(aj[["PEF", "DT_AJUIZAMENTO", "NOMECOMARCA"]].head(3))


mid-window rows 768 | unique PEF 768 | unique ID_DEBITO 768
mid-window year counts:
DT_AJUIZAMENTO
2017    143
2018    409
2019     73
2020    143
saved /Volumes/Meedi_Etore_HD1/CEMEPI/dumps/cemepi_api/extracao=2026-03/samples/toy_ajuizamento_midwindow.parquet


debito rows 300 | unique ID_DEBITO 300 | saved toy_debito_sample_v2.parquet
ID_DEBITO overlap (mid-window aj × debito sample): 0
Note: independent API heads often yield 0 overlap; full dump join is future work.


,PEF,DT_AJUIZAMENTO,NOMECOMARCA
1,1500004-47.2017.8.26.0040,2017-03-20,Comarca de Américo Brasiliense
6,1500644-22.2017.8.26.0114,2017-04-01,Comarca de Campinas
9,1526343-06.2017.8.26.0602,2017-10-25,Comarca de Sorocaba


## 5. Universos A vs B — formatos CNJ (diagnóstico de match baixo)

### Método — exact match (referência)
Join por igualdade de string entre chaves nominais ($PEF = FACE.numero$). Em *record linkage*, isto é o degrau “exact match” antes de regras de normalização (Fellegi & Sunter, 1969 — framing clássico; **aqui não** estimamos pesos probabilísticos).

$$
M_{\mathrm{exact}} = \sum_{i=1}^{N}\mathbf{1}\{PEF_i \in \mathcal{N}_{\mathrm{FACE}}\}, \qquad
\hat{c}_{\mathrm{exact}} = M_{\mathrm{exact}} / N.
$$

### Método — normalização CNJ (Christen, 2012)
Seja $\delta(s)$ a projeção digits-only e $\mu(d)$ a máscara CNJ `NNNNNNN-DD.AAAA.J.TR.OOOO` a partir de 20 dígitos (zero-pad se $|d|<20$). O degrau normalizado é a união:

$$
\hat{c}_{\mathrm{norm}} = \frac{1}{N}\sum_{i=1}^{N}
\mathbf{1}\bigl\{
PEF_i \in \mathcal{N}
\;\lor\;
\delta(PEF_i) \in \mathcal{D}
\;\lor\;
\mu(\delta(PEF_i)) \in \mathcal{N}
\bigr\},
$$

onde $\mathcal{N}$ é o conjunto de `numero` e $\mathcal{D}$ o de formas 20 dígitos (alinhado a `num_processo_limpo`). Lift de normalização: $\Delta = \hat{c}_{\mathrm{norm}} - \hat{c}_{\mathrm{exact}}$.

### Interpretação rigorosa — Figuras 5a / 5b (antes de plotar)
- **Figura 5a (histograma agrupado):** eixo $x$ = comprimento em dígitos $|\delta(s)|$; cor = universo (A = dump PEF mid-window; B = FACE `numero` LIMIT-$k$). Altura = contagem. Pico em 20 em ambos os lados ⇒ formatos alinhados.
- **Figura 5b:** eixo $x$ = `len(string)` bruta; máscara CNJ típica ≈ 25 caracteres.
- **Viés de seleção:** B é `LIMIT` (não amostra aleatória do lake); serve só para *sanity check* de formato, não para estimar cobertura populacional do scrape.
- Se formatos coincidem e $\hat{c}_{\mathrm{exact}}$ permanece baixo, o gargalo tende a ser **esparsidade/cobertura do lake**, não máscara de string.


In [4]:
def digits_only(s: str) -> str:
    return re.sub(r"\D", "", s or "")


def cnj_mask_from_digits(digits: str) -> str | None:
    """CNJ mask NNNNNNN-DD.AAAA.J.TR.OOOO from up to 20 digits (zfill if shorter)."""
    if not digits:
        return None
    d = digits.zfill(20) if len(digits) < 20 else digits
    if len(d) != 20:
        return None
    return f"{d[0:7]}-{d[7:9]}.{d[9:13]}.{d[13]}.{d[14:16]}.{d[16:20]}"


# --- Universe A: all mid-window PEFs (format audit; join set chosen later) ---
pef_all = (
    aj["PEF"].dropna().astype(str).str.strip()
    .loc[lambda s: s.str.len() > 0]
    .drop_duplicates()
)
univ_a = pd.DataFrame({"pef": pef_all.tolist()})
univ_a["char_len"] = univ_a["pef"].str.len()
univ_a["digits"] = univ_a["pef"].map(digits_only)
univ_a["digit_len"] = univ_a["digits"].str.len()
univ_a["cnj_mask"] = univ_a["digits"].map(cnj_mask_from_digits)

print("Universe A (dump PEF mid-window): n_unique =", len(univ_a))
print("char_len:\n", univ_a["char_len"].value_counts().sort_index().to_string())
print("digit_len:\n", univ_a["digit_len"].value_counts().sort_index().to_string())
n_anom_a = int((univ_a["digit_len"] != 20).sum())
print(f"format anomalies (digit_len != 20): {n_anom_a}")

# --- Universe B: FACE numero format probe (LIMIT only — not full scan / not join) ---
con_fmt = duckdb.connect()
con_fmt.execute(f"SET memory_limit='{DUCKDB_MEM}'")
face_path_sql = str(SILVER_FACE_CLEAN).replace("'", "''")
univ_b = con_fmt.execute(f"""
SELECT CAST(numero AS VARCHAR) AS numero,
       length(CAST(numero AS VARCHAR)) AS char_len,
       length(regexp_replace(CAST(numero AS VARCHAR), '[^0-9]', '', 'g')) AS digit_len
FROM delta_scan('{face_path_sql}')
WHERE numero IS NOT NULL
LIMIT {int(FACE_FORMAT_SAMPLE)}
""").fetchdf()
con_fmt.close()

print(f"\nUniverse B (FACE numero LIMIT {FACE_FORMAT_SAMPLE}): n =", len(univ_b))
print("char_len:\n", univ_b["char_len"].value_counts().sort_index().to_string())
print("digit_len:\n", univ_b["digit_len"].value_counts().sort_index().to_string())

fmt_cmp = pd.DataFrame({
    "side": (["dump_PEF"] * len(univ_a)) + (["face_LIMIT"] * len(univ_b)),
    "digit_len": list(univ_a["digit_len"]) + list(univ_b["digit_len"].astype(int)),
    "char_len": list(univ_a["char_len"]) + list(univ_b["char_len"].astype(int)),
})
fig_fmt = px.histogram(
    fmt_cmp, x="digit_len", color="side", barmode="group",
    title="Fig. 5a — Universos A vs B: comprimento em dígitos (PEF mid-window vs FACE LIMIT)",
    labels={"digit_len": "N de dígitos |δ(s)|", "side": "Universo", "count": "Contagem"},
)
show_and_save(fig_fmt, "fig05a_digit_len_hist", height=400)

fig_char = px.histogram(
    fmt_cmp, x="char_len", color="side", barmode="group",
    title="Fig. 5b — Universos A vs B: comprimento em caracteres (string bruta)",
    labels={"char_len": "len(string)", "side": "Universo", "count": "Contagem"},
)
show_and_save(fig_char, "fig05b_char_len_hist", height=400)


Universe A (dump PEF mid-window): n_unique = 768
char_len:
 char_len
25    755
29     13
digit_len:
 digit_len
20    755
24     13
format anomalies (digit_len != 20): 13



Universe B (FACE numero LIMIT 500): n = 500
char_len:
 char_len
25    500
digit_len:
 digit_len
20    500


static → fig05a_digit_len_hist.png, fig05a_digit_len_hist.svg


static → fig05b_char_len_hist.png, fig05b_char_len_hist.svg


PosixPath('/Users/etorebraga/Code/cemepi-ctf-intel-fiscal/projects/litigancia/notebooks/playground/output/pef_toy_figures/fig05b_char_len_hist.png')

## 6. Escada de match — exact → digits-only → máscara CNJ

### Método — amostra estratificada (join set)
Em vez de `head(N)` (viesado pela ordem da API), alocamos até $N_{\mathrm{cap}}=PEF\_JOIN\_CAP$ PEFs **proporcionalmente por ano** na janela (seed fixo):

$$
n_y = \mathrm{round}\!\left(N_{\mathrm{cap}}\cdot\frac{N_y}{\sum_{y'} N_{y'}}\right),
\quad \sum_y n_y = N_{\mathrm{cap}},
$$

com ajuste guloso para fechar a soma. DuckDB: `numero` / dígitos com `IN (≤250)`.

### Método — coverage rate (descritiva)
Em cada degrau $r\in\{\mathrm{exact},\,\mathrm{digits},\,\mathrm{mask},\,\mathrm{norm}\}$:

$$
\hat{c}_r = \frac{M_r}{N},\qquad
M_{\mathrm{norm}} = \bigl|\{i:\; \text{exact}_i \lor \text{digits}_i \lor \text{mask}_i \lor \text{zpad}_i\}\bigr|.
$$

Números **sempre** recomputados nesta célula (não inventados).

### Interpretação — sem figura dedicada nesta célula
A tabela/prints da escada são o resultado principal. Se $\hat{c}_{\mathrm{exact}}\approx\hat{c}_{\mathrm{norm}}$ e os histogramas 5a/5b coincidem, a baixa taxa **não** se explica por pontuação/hífen; aponta para PEFs ausentes no scrape FACE ou composição amostral. Caveat: taxa `head(250)` (ordem da API) é viesada e só reportada como comparação.


In [5]:
# --- Stratified join set by year (reproducible) ---
aj_u = aj.dropna(subset=["PEF"]).copy()
aj_u["PEF"] = aj_u["PEF"].astype(str).str.strip()
aj_u = aj_u.loc[aj_u["PEF"].str.len() > 0].drop_duplicates(subset=["PEF"], keep="first")
aj_u["year"] = aj_u["DT_AJUIZAMENTO"].dt.year
vc_year = aj_u["year"].value_counts().sort_index()
alloc = ((vc_year / vc_year.sum()) * PEF_JOIN_CAP).round().astype(int)
while int(alloc.sum()) > PEF_JOIN_CAP:
    alloc.iloc[int(alloc.argmax())] -= 1
while int(alloc.sum()) < PEF_JOIN_CAP and int(alloc.sum()) < len(aj_u):
    rem = vc_year - alloc
    if int(rem.max()) <= 0:
        break
    alloc.iloc[int(rem.argmax())] += 1

parts = []
for y, n in alloc.items():
    pool = aj_u.loc[aj_u["year"] == y]
    parts.append(pool.sample(n=min(int(n), len(pool)), random_state=RANDOM_STATE))
aj_join = pd.concat(parts, ignore_index=True)

pef_df = pd.DataFrame({"pef": aj_join["PEF"].tolist()})
pef_df["pef_digits"] = pef_df["pef"].map(digits_only)
pef_df["pef_mask"] = pef_df["pef_digits"].map(cnj_mask_from_digits)
pef_df["digit_len"] = pef_df["pef_digits"].str.len()
n_pef = len(pef_df)
assert n_pef <= PEF_JOIN_CAP

print("join-set allocation by year:", alloc.to_dict(), "| n_pef =", n_pef)
print("join-set year counts:", aj_join["year"].value_counts().sort_index().to_dict())
print("join-set digit_len anomalies:", int((pef_df["digit_len"] != 20).sum()))

con = duckdb.connect()
con.execute(f"SET memory_limit='{DUCKDB_MEM}'")
con.register("pef_list", pef_df[["pef", "pef_digits", "pef_mask"]])
face_path = str(SILVER_FACE_CLEAN).replace("'", "''")

face_hit = con.execute(f"""
SELECT DISTINCT
  CAST(numero AS VARCHAR) AS numero,
  regexp_replace(CAST(numero AS VARCHAR), '[^0-9]', '', 'g') AS digits,
  cd_processo, foro, vara, classe
FROM delta_scan('{face_path}')
WHERE CAST(numero AS VARCHAR) IN (SELECT pef FROM pef_list)
   OR regexp_replace(CAST(numero AS VARCHAR), '[^0-9]', '', 'g') IN (SELECT pef_digits FROM pef_list)
""").fetchdf()

face_by_num = {str(r["numero"]): r for r in face_hit.to_dict("records")}
face_by_dig: dict[str, dict] = {}
for r in face_hit.to_dict("records"):
    face_by_dig.setdefault(str(r["digits"]), r)

ladder_rows = []
for r in pef_df.itertuples():
    exact = r.pef in face_by_num
    dig = r.pef_digits in face_by_dig
    mask = bool(r.pef_mask) and (r.pef_mask in face_by_num)
    zpad = False
    if 0 < len(r.pef_digits) < 20:
        m = cnj_mask_from_digits(r.pef_digits)
        zpad = bool(m) and (m in face_by_num or r.pef_digits.zfill(20) in face_by_dig)
    any_norm = bool(exact or dig or mask or zpad)
    hit = face_by_num.get(r.pef) or face_by_dig.get(r.pef_digits) or (
        face_by_num.get(r.pef_mask) if r.pef_mask else None
    )
    ladder_rows.append({
        "pef": r.pef,
        "exact": exact,
        "digits_only": dig,
        "mask_hit": mask,
        "zpad_hit": zpad,
        "normalized": any_norm,
        "face_numero": (hit or {}).get("numero"),
        "face_cd_processo": (hit or {}).get("cd_processo"),
        "face_foro": (hit or {}).get("foro"),
        "face_vara": (hit or {}).get("vara"),
        "face_classe": (hit or {}).get("classe"),
    })

ladder = pd.DataFrame(ladder_rows)
n_exact = int(ladder["exact"].sum())
n_norm = int(ladder["normalized"].sum())
rate_exact = n_exact / n_pef if n_pef else float("nan")
rate_norm = n_norm / n_pef if n_pef else float("nan")
n_digits_beyond = int((ladder["digits_only"] & ~ladder["exact"]).sum())
n_mask_beyond = int((ladder["mask_hit"] & ~ladder["exact"]).sum())

print(f"Window: {WINDOW_START.date()} … {WINDOW_END.date()}")
print(f"PEFs queried (stratified mid-window): {n_pef}")
print(f"FACE rows returned by IN probe: {len(face_hit)}")
print("--- MATCH LADDER (computed) ---")
print(f"exact PEF<->numero:           {n_exact}/{n_pef} = {rate_exact:.1%}")
print(f"digits-only:                {int(ladder['digits_only'].sum())}/{n_pef} = {ladder['digits_only'].mean():.1%}")
print(f"mask hit (raw):             {int(ladder['mask_hit'].sum())}/{n_pef} = {ladder['mask_hit'].mean():.1%}")
print(f"normalized (union):         {n_norm}/{n_pef} = {rate_norm:.1%}")
print(f"digits beyond exact: {n_digits_beyond} | mask beyond exact: {n_mask_beyond}")

if n_exact:
    cd_equals_pef = float(
        (ladder.loc[ladder["exact"], "pef"] == ladder.loc[ladder["exact"], "face_cd_processo"]).mean()
    )
    sample_ids = ladder.loc[ladder["exact"], ["pef", "face_cd_processo", "face_foro"]].head(3)
else:
    cd_equals_pef = float("nan")
    sample_ids = ladder.head(0)
print(f"Among exact matched, share where pef == cd_processo: {cd_equals_pef}")
display(sample_ids)

# prior head(250) caveat number from v2 — recomputed briefly for comparison note
pef_head = (
    aj["PEF"].dropna().astype(str).str.strip()
    .loc[lambda s: s.str.len() > 0].drop_duplicates().head(PEF_JOIN_CAP)
)
con.register("pef_head", pd.DataFrame({"pef": pef_head.tolist()}))
n_head_match = int(con.execute(f"""
SELECT COUNT(*) AS n FROM (
  SELECT p.pef FROM pef_head p
  INNER JOIN (
    SELECT DISTINCT CAST(numero AS VARCHAR) AS numero
    FROM delta_scan('{face_path}')
    WHERE CAST(numero AS VARCHAR) IN (SELECT pef FROM pef_head)
  ) f ON p.pef = f.numero
)
""").fetchone()[0])
rate_head = n_head_match / len(pef_head)
print(f"Caveat head(250) exact (order-biased, v2): {n_head_match}/{len(pef_head)} = {rate_head:.1%}")

cov = ladder.rename(columns={"normalized": "matched_norm", "exact": "matched"})[
    ["pef", "matched", "matched_norm", "digits_only", "mask_hit", "zpad_hit",
     "face_numero", "face_cd_processo", "face_foro", "face_vara", "face_classe"]
].copy()
aj_cov = aj_join.merge(cov, left_on="PEF", right_on="pef", how="left")
aj_cov["matched"] = aj_cov["matched"].fillna(False).astype(bool)
aj_cov["matched_norm"] = aj_cov["matched_norm"].fillna(False).astype(bool)
match_rate = rate_exact
match_rate_norm = rate_norm
n_matched = n_exact


join-set allocation by year: {2017: 47, 2018: 132, 2019: 24, 2020: 47} | n_pef = 250
join-set year counts: {2017: 47, 2018: 132, 2019: 24, 2020: 47}
join-set digit_len anomalies: 4


Window: 2017-01-01 … 2020-12-31
PEFs queried (stratified mid-window): 250
FACE rows returned by IN probe: 6
--- MATCH LADDER (computed) ---
exact PEF<->numero:           6/250 = 2.4%
digits-only:                6/250 = 2.4%
mask hit (raw):             6/250 = 2.4%
normalized (union):         6/250 = 2.4%
digits beyond exact: 0 | mask beyond exact: 0
Among exact matched, share where pef == cd_processo: 0.0


,pef,face_cd_processo,face_foro
12,1501482-21.2017.8.26.0451,CJ000FQBA0000,Foro de Piracicaba
37,1505096-84.2017.8.26.0014,0E0008V1A0000,Foro das Execuções Fiscais Estaduais
45,1511997-67.2017.8.26.0564,FO00070V00000,Foro de São Bernardo do Campo


Caveat head(250) exact (order-biased, v2): 11/250 = 4.4%


## 7. Taxas estratificadas + auditoria de unmatched (sem PII)

### Método — cobertura estratificada
Para estrato $s$ (ano de ajuizamento):

$$
\hat{c}_s = \frac{M_s}{n_s},\qquad
\mathrm{Var}_{\mathrm{toy}}(\hat{c}_s) \text{ alta quando } n_s \text{ é pequeno}.
$$

Reportamos $\hat{c}_s^{\mathrm{exact}}$ e $\hat{c}_s^{\mathrm{norm}}$ lado a lado. Auditoria de unmatched: só **contagens** de anomalias de formato ($|\delta|\neq 20$, `len≠25`) — sem listar identificadores completos.

### Interpretação rigorosa — Figura 7 (antes de plotar)
- **Eixos:** $x$ = ano de ajuizamento; $y$ = taxa $\hat{c}_s$ (escala percentual).
- **Codificação:** barras agrupadas exact vs normalizado (duas séries).
- **Leitura:** se as duas séries coincidem em todos os anos, lift de normalização ≈ 0 por estrato.
- **Viés / cautela:** $n_s$ desigual (ex.: 2018 dominante no probe); intervalos não plotados — toy, não inferência estadual. Diferenças grandes entre anos com formatos alinhados reforçam hipótese de **cobertura desigual do scrape**.


In [6]:
# --- By year ---
by_year = (
    aj_cov.groupby(aj_cov["DT_AJUIZAMENTO"].dt.year)
    .agg(n_pef=("PEF", "size"), n_exact=("matched", "sum"), n_norm=("matched_norm", "sum"))
    .assign(
        rate_exact=lambda x: x["n_exact"] / x["n_pef"],
        rate_norm=lambda x: x["n_norm"] / x["n_pef"],
    )
)
print("Stratified by year:")
display(by_year)

fig_year_rate = go.Figure()
fig_year_rate.add_bar(name="exact", x=by_year.index.astype(str), y=by_year["rate_exact"])
fig_year_rate.add_bar(name="normalized", x=by_year.index.astype(str), y=by_year["rate_norm"])
fig_year_rate.update_layout(
    barmode="group",
    title="Fig. 7 — Taxa de match por ano (exact vs normalizado) — join set estratificado",
    yaxis_tickformat=".0%",
    yaxis_title="cobertura descritiva ĉ_s",
    xaxis_title="Ano de ajuizamento",
    legend_title="Degrau",
)
show_and_save(fig_year_rate, "fig07_year_rate_exact_vs_norm", height=400)

# --- Unmatched format audit (counts only) ---
unm = aj_cov.loc[~aj_cov["matched"]].copy()
unm["digit_len"] = unm["PEF"].map(lambda s: len(digits_only(str(s))))
unm["char_len"] = unm["PEF"].astype(str).str.len()
audit = {
    "n_unmatched": int(len(unm)),
    "digit_len_ne_20": int((unm["digit_len"] != 20).sum()),
    "char_len_ne_25": int((unm["char_len"] != 25).sum()),
    "mask_build_fail": int(unm["PEF"].map(lambda s: cnj_mask_from_digits(digits_only(str(s))) is None).sum()),
}
print("Unmatched format audit (counts, no PII):", audit)
print("unmatched digit_len distribution:\n", unm["digit_len"].value_counts().sort_index().to_string())


Stratified by year:


,n_pef,n_exact,n_norm,rate_exact,rate_norm
DT_AJUIZAMENTO,,,,,
2017,47,3,3,0.063830,0.063830
2018,132,1,1,0.007576,0.007576
2019,24,0,0,0.000000,0.000000
2020,47,2,2,0.042553,0.042553


static → fig07_year_rate_exact_vs_norm.png, fig07_year_rate_exact_vs_norm.svg
Unmatched format audit (counts, no PII): {'n_unmatched': 244, 'digit_len_ne_20': 4, 'char_len_ne_25': 4, 'mask_build_fail': 4}
unmatched digit_len distribution:
 digit_len
20    240
24      4


## 8. Tops — comarcas, anos, campos lake anonimizados (matched only)

### Método — contagens e heatmap
Tops = $\mathrm{arg\,top}_k$ das frequências empíricas no join set (ou subset matched). Heatmap: tabela de contingência comarca × ano

$$
H_{c,y} = \#\{i:\; \mathrm{comarca}_i=c,\; \mathrm{year}_i=y\}
$$

restrita às top-$k$ comarcas do join set. Cores = $H_{c,y}$; zeros = ausência no *join set* (não ausência populacional).

### Interpretação rigorosa — Figuras 8a–8e (antes de plotar)
- **Fig. 8a (barras anuais):** $x$ = ano; $y$ = $n$ PEFs no join set estratificado — espelha a alocação $n_y$, não a população estadual.
- **Fig. 8b (barras horizontais):** $y$ = comarca; $x$ = contagem no join set. Top-$k$ por frequência; ordem crescente no eixo.
- **Fig. 8c (rosca) / 8d (barras):** partição matched vs unmatched; título traz $\hat{c}_{\mathrm{exact}}$ e $\hat{c}_{\mathrm{norm}}$. **Seleção:** denominador = join set ($N\le 250$), não o lake inteiro.
- **Fig. 8e (timeline mensal empilhada):** $x$ = ano-mês; cor = status de match; útil para ver concentração temporal dos poucos hits.
- **Fig. 8f (heatmap):** top comarcas × ano; células escuras = mais PEFs no estrato do *slice*.
- Tops de `foro` / `classe` (tabelas): **somente matched** — campos administrativos do lake sem PII; $n$ matched pequeno ⇒ ranks instáveis.


In [7]:
# Top years (join set)
year_vc = (
    aj_cov["DT_AJUIZAMENTO"].dt.year.value_counts().sort_index()
    .rename_axis("year").reset_index(name="n_pef")
)
fig_years = px.bar(
    year_vc, x="year", y="n_pef",
    title=f"Fig. 8a — Ajuizamentos por ano no join set estratificado (N={n_pef})",
    labels={"n_pef": "PEFs distintos", "year": "Ano de ajuizamento"},
)
show_and_save(fig_years, "fig08a_years_joinset", height=400)

# Top comarcas: dump window (full mid-window), matched, unmatched
def top_comarca_table(df: pd.DataFrame, label: str, k: int = 12) -> pd.DataFrame:
    t = (
        df["NOMECOMARCA"].fillna("(missing)").value_counts().head(k)
        .rename_axis("comarca").reset_index(name="n_pef")
        .assign(slice=label)
    )
    return t

top_window = top_comarca_table(aj, "mid-window dump")
top_join = top_comarca_table(aj_cov, "join set")
top_m = top_comarca_table(aj_cov.loc[aj_cov["matched"]], "matched")
top_u = top_comarca_table(aj_cov.loc[~aj_cov["matched"]], "unmatched")

print("Top comarcas — mid-window dump:")
display(top_window)
print("Top comarcas — matched:")
display(top_m)
print("Top comarcas — unmatched:")
display(top_u)

fig_com = px.bar(
    top_join, x="n_pef", y="comarca", orientation="h",
    title=f"Fig. 8b — Top comarcas no join set (janela {WINDOW_START.year}–{WINDOW_END.year})",
    labels={"n_pef": "PEFs", "comarca": "Comarca"},
)
fig_com.update_layout(yaxis={"categoryorder": "total ascending"})
show_and_save(fig_com, "fig08b_top_comarcas_joinset", height=440)

# Matched vs unmatched pie/bar
match_vc = (
    aj_cov["matched"].map({True: "matched", False: "unmatched"})
    .value_counts().rename_axis("status").reset_index(name="n")
)
fig_pie = px.pie(
    match_vc, names="status", values="n",
    title=f"Fig. 8c — Cobertura exact PEF↔FACE.numero (N={n_pef}, ĉ={match_rate:.1%}; norm={match_rate_norm:.1%})",
    hole=0.35,
)
show_and_save(fig_pie, "fig08c_match_donut", height=400)

fig_match_bar = px.bar(
    match_vc, x="status", y="n", text="n",
    title="Fig. 8d — Matched vs unmatched (contagens, exact)",
    labels={"n": "PEFs", "status": "Status"},
)
show_and_save(fig_match_bar, "fig08d_match_counts", height=380)

# Timeline
tl = aj_cov.copy()
tl["ym"] = tl["DT_AJUIZAMENTO"].dt.to_period("M").astype(str)
tl_vc = tl.groupby(["ym", "matched"]).size().rename("n").reset_index()
tl_vc["status"] = tl_vc["matched"].map({True: "matched", False: "unmatched"})
fig_tl = px.bar(
    tl_vc, x="ym", y="n", color="status",
    title=f"Fig. 8e — Timeline mensal de ajuizamentos no join set ({WINDOW_START.year}–{WINDOW_END.year})",
    labels={"ym": "Mês", "n": "PEFs"},
)
fig_tl.update_layout(xaxis_tickangle=-45)
show_and_save(fig_tl, "fig08e_timeline_monthly", height=420)

# Heatmap comarca x year
top_com_names = set(top_join["comarca"].head(8))
hm = aj_cov.copy()
hm["comarca"] = hm["NOMECOMARCA"].fillna("(missing)")
hm = hm.loc[hm["comarca"].isin(top_com_names)]
hm["year"] = hm["DT_AJUIZAMENTO"].dt.year
ct = hm.pivot_table(index="comarca", columns="year", values="PEF", aggfunc="count", fill_value=0)
fig_hm = px.imshow(
    ct, text_auto=True, aspect="auto",
    title="Fig. 8f — Heatmap: top comarcas × ano (join set)",
    labels=dict(color="n PEFs"),
)
show_and_save(fig_hm, "fig08f_heatmap_comarca_year", height=440)

# Lake anonymized fields — matched only
if n_matched:
    print("Top FACE.foro (matched only, anonymized administrative):")
    display(aj_cov.loc[aj_cov["matched"], "face_foro"].fillna("(missing)").value_counts().head(10).rename_axis("foro").reset_index(name="n"))
    print("Top FACE.classe (matched only):")
    display(aj_cov.loc[aj_cov["matched"], "face_classe"].fillna("(missing)").value_counts().head(10).rename_axis("classe").reset_index(name="n"))
else:
    print("No matched PEFs — skip lake field tops.")


static → fig08a_years_joinset.png, fig08a_years_joinset.svg
Top comarcas — mid-window dump:


,comarca,n_pef,slice
0,Vara das Execuções Fiscais Estaduais da Comarc...,570,mid-window dump
1,Comarca de Barueri,23,mid-window dump
2,Comarca de Guarulhos,18,mid-window dump
3,Comarca de São Bernardo do Campo,13,mid-window dump
4,Comarca de Campinas,11,mid-window dump
5,Comarca de Osasco,10,mid-window dump
6,Comarca de Taubaté,7,mid-window dump
7,Comarca de Mauá,7,mid-window dump
8,Comarca de São José dos Campos,6,mid-window dump
9,Comarca de Cotia,5,mid-window dump


Top comarcas — matched:


,comarca,n_pef,slice
0,Vara das Execuções Fiscais Estaduais da Comarc...,2,matched
1,Comarca de Piracicaba,1,matched
2,Comarca de São Bernardo do Campo,1,matched
3,Comarca de Barueri,1,matched
4,Comarca de Itapetininga,1,matched


Top comarcas — unmatched:


...

static → fig08e_timeline_monthly.png, fig08e_timeline_monthly.svg


static → fig08f_heatmap_comarca_year.png, fig08f_heatmap_comarca_year.svg
Top FACE.foro (matched only, anonymized administrative):


,foro,n
0,Foro das Execuções Fiscais Estaduais,2
1,Foro de Piracicaba,1
2,Foro de São Bernardo do Campo,1
3,Foro de Barueri,1
4,Foro de Itapetininga,1


Top FACE.classe (matched only):


,classe,n
0,Execução Fiscal,6


## 9. Grafos toy (NetworkX + Plotly) — layouts legíveis, só tops

### Método — grafo bipartido descritivo
Seja $G=(V_C \cup V_P, E)$ bipartido, com $V_C$ = comarcas (top-$K$) e $V_P$ = PEFs truncados (últimos 4 dígitos apenas). Arestas $e=\{c,p\}$ existem quando o PEF no join set pertence à comarca $c$. Layout em duas colunas via `networkx.bipartite_layout` (Hagberg, Schult & Swart, 2008 — NetworkX; formulação clássica em Asratian et al., 1998). **Uso exploratório / dataviz** — não é rede social de partes/OAB nem modelo estatístico de grafo.

Tamanho do nó $\propto 10 + \alpha\cdot\deg(v)$; cor = partição / status de match. Isolados omitidos. Small-multiples separam matched vs unmatched (cap ≤3 unmatched/comarca para legibilidade).

### Interpretação rigorosa — Figuras 9a–9d (antes de plotar)
- **Fig. 9a (bipartido misto):** coluna esquerda = comarcas; direita = PEF last-4; azul = matched, vermelho = unmatched amostrado, laranja = comarca. Hover = rótulo + grau.
- **Fig. 9b / 9c (small multiples):** mesmo encoding, filtrado só matched / só unmatched — evita misturar densidades.
- **Fig. 9d (comarca ↔ status):** nós de status fixos no eixo vertical; spring layout com $k$ elevado; espessura implícita via peso da aresta (contagem). Mostra quais top comarcas tocam o lado matched.
- **Viés de seleção:** só top-$K=15$ comarcas; unmatched capped; last-4 pode colidir (homografia visual) — labels não são IDs únicos globais.


In [8]:
def plot_nx_plotly(
    G: nx.Graph,
    pos: dict,
    title: str,
    *,
    color_attr: str = "kind",
    color_map: dict | None = None,
    size_scale: float = 3.0,
    height: int = 560,
    legend_kinds: list[str] | None = None,
):
    """NetworkX layout → Plotly scatter+edges (interactive + Kaleido-exportable)."""
    edge_x, edge_y = [], []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
    traces: list = [
        go.Scatter(
            x=edge_x, y=edge_y, mode="lines",
            line=dict(width=0.6, color="#B0B0B0"), hoverinfo="none", showlegend=False,
        )
    ]

    kinds = legend_kinds or sorted({G.nodes[n].get(color_attr, "node") for n in G.nodes()})
    cmap = color_map or {
        "pef": "#4C78A8", "comarca": "#F58518", "status": "#54A24B",
        "pef_matched": "#4C78A8", "pef_unmatched": "#E45756",
    }
    for kind in kinds:
        nodes = [n for n in G.nodes() if G.nodes[n].get(color_attr) == kind]
        if not nodes:
            continue
        node_x, node_y, node_text, node_size = [], [], [], []
        for n in nodes:
            x, y = pos[n]
            node_x.append(x)
            node_y.append(y)
            label = G.nodes[n].get("label", str(n))
            deg = G.degree(n)
            node_text.append(f"{label}<br>kind={kind}<br>degree={deg}")
            node_size.append(10 + size_scale * deg)
        traces.append(go.Scatter(
            x=node_x, y=node_y, mode="markers", name=str(kind),
            marker=dict(
                size=node_size, color=cmap.get(kind, "#9D755D"),
                line=dict(width=0.5, color="#333"),
            ),
            text=node_text, hoverinfo="text",
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=title, height=height,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor="x", scaleratio=1),
        margin=dict(l=10, r=10, t=50, b=10),
        plot_bgcolor="white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    return fig


TOP_N_COMARCAS_GRAPH = 15

# --- 9.1 Bipartite PEF <-> comarca on top comarcas only ---
com_counts = aj_cov["NOMECOMARCA"].fillna("(missing)").value_counts()
top_coms_g = list(com_counts.head(TOP_N_COMARCAS_GRAPH).index)
bip_src = aj_cov.loc[aj_cov["NOMECOMARCA"].fillna("(missing)").isin(top_coms_g)].copy()
bip_m = bip_src.loc[bip_src["matched"]]
bip_u = bip_src.loc[~bip_src["matched"]]
u_cap = []
for com, g in bip_u.groupby(bip_u["NOMECOMARCA"].fillna("(missing)")):
    u_cap.append(g.head(3))
bip_draw = pd.concat([bip_m, *u_cap], ignore_index=True) if len(u_cap) else bip_m.copy()
bip_draw = bip_draw.drop_duplicates(subset=["PEF"])

Bip = nx.Graph()
left, right = [], []
for _, row in bip_draw.iterrows():
    pef = str(row["PEF"])
    pef_node = f"pef:{pef[-4:]}"  # last-4 only
    com = str(row["NOMECOMARCA"] or "(missing)")
    com_short = com if len(com) <= 42 else com[:39] + "..."
    com_node = f"com:{com_short}"
    kind_pef = "pef_matched" if bool(row["matched"]) else "pef_unmatched"
    if pef_node not in Bip:
        Bip.add_node(pef_node, kind=kind_pef, label=pef_node, bipartite=1)
        right.append(pef_node)
    if com_node not in Bip:
        Bip.add_node(com_node, kind="comarca", label=com_short, bipartite=0)
        left.append(com_node)
    Bip.add_edge(pef_node, com_node)

isolates = list(nx.isolates(Bip))
Bip.remove_nodes_from(isolates)
left = [n for n in left if n in Bip]
right = [n for n in right if n in Bip]

if Bip.number_of_nodes() >= 2 and left and right:
    pos_b = nx.bipartite_layout(Bip, nodes=left, align="vertical", scale=2.0, aspect_ratio=0.55)
    for side_nodes, x_target in ((left, -1.0), (right, 1.0)):
        side_nodes_sorted = sorted(side_nodes, key=lambda n: -Bip.degree(n))
        for i, n in enumerate(side_nodes_sorted):
            y = 1.0 - 2.0 * (i / max(len(side_nodes_sorted) - 1, 1))
            pos_b[n] = (x_target, y)
    fig_bip = plot_nx_plotly(
        Bip, pos_b,
        title=f"Fig. 9a — Bipartido PEF↔comarca (top {TOP_N_COMARCAS_GRAPH}; PEF=last-4; matched+amostra unmatched)",
        color_map={"comarca": "#F58518", "pef_matched": "#4C78A8", "pef_unmatched": "#E45756"},
        legend_kinds=["comarca", "pef_matched", "pef_unmatched"],
        size_scale=2.5, height=640,
    )
    show_and_save(fig_bip, "fig09a_bipartite_pef_comarca", height=640, width=1000)
    print(f"Bipartite: nodes={Bip.number_of_nodes()} edges={Bip.number_of_edges()} isolates_removed={len(isolates)}")
    deg_tbl = (
        pd.DataFrame({
            "node": list(Bip.nodes),
            "degree": [Bip.degree(n) for n in Bip.nodes],
            "kind": [Bip.nodes[n].get("kind") for n in Bip.nodes],
        })
        .sort_values("degree", ascending=False).head(12)
    )
    print("Top degree nodes:")
    display(deg_tbl)
else:
    print("Bipartite graph skipped — insufficient nodes after filters.")

# --- 9.2 Small multiples: matched-only vs unmatched-only ---
for flag, title_suf, color_pef, stem in [
    (True, "só matched", "pef_matched", "fig09b_bipartite_matched_only"),
    (False, "só unmatched (≤3 PEF/comarca)", "pef_unmatched", "fig09c_bipartite_unmatched_only"),
]:
    src = bip_src.loc[bip_src["matched"] == flag]
    if not flag:
        caps = [g.head(3) for _, g in src.groupby(src["NOMECOMARCA"].fillna("(missing)"))]
        src = pd.concat(caps, ignore_index=True) if caps else src.head(0)
    G2 = nx.Graph()
    L2, R2 = [], []
    for _, row in src.iterrows():
        pef_node = f"pef:{str(row['PEF'])[-4:]}"
        com = str(row["NOMECOMARCA"] or "(missing)")
        com_short = com if len(com) <= 42 else com[:39] + "..."
        com_node = f"com:{com_short}"
        if pef_node not in G2:
            G2.add_node(pef_node, kind=color_pef, label=pef_node, bipartite=1)
            R2.append(pef_node)
        if com_node not in G2:
            G2.add_node(com_node, kind="comarca", label=com_short, bipartite=0)
            L2.append(com_node)
        G2.add_edge(pef_node, com_node)
    G2.remove_nodes_from(list(nx.isolates(G2)))
    L2 = [n for n in L2 if n in G2]
    R2 = [n for n in R2 if n in G2]
    if G2.number_of_nodes() < 2 or not L2 or not R2:
        print(f"Small multiple ({title_suf}): skipped")
        continue
    pos2 = nx.bipartite_layout(G2, nodes=L2, align="vertical", scale=2.0, aspect_ratio=0.55)
    for side_nodes, x_target in ((L2, -1.0), (R2, 1.0)):
        side_sorted = sorted(side_nodes, key=lambda n: -G2.degree(n))
        for i, n in enumerate(side_sorted):
            y = 1.0 - 2.0 * (i / max(len(side_sorted) - 1, 1))
            pos2[n] = (x_target, y)
    fig2 = plot_nx_plotly(
        G2, pos2,
        title=f"Fig. 9{'b' if flag else 'c'} — Bipartido PEF↔comarca — {title_suf}",
        color_map={"comarca": "#F58518", color_pef: "#4C78A8" if flag else "#E45756"},
        legend_kinds=["comarca", color_pef],
        height=520, size_scale=2.8,
    )
    show_and_save(fig2, stem, height=520, width=1000)
    print(f"Small multiple ({title_suf}): nodes={G2.number_of_nodes()} edges={G2.number_of_edges()}")

# --- 9.3 Comarca <-> match-status (top comarcas, spring with high k) ---
Sg = nx.Graph()
top_set = set(top_coms_g)
for comarca, g in aj_cov.groupby(aj_cov["NOMECOMARCA"].fillna("(missing)")):
    if comarca not in top_set:
        continue
    label = comarca if len(comarca) <= 42 else comarca[:39] + "..."
    cnode = f"com:{label}"
    Sg.add_node(cnode, kind="comarca", label=label)
    n_m = int(g["matched"].sum())
    n_u = int((~g["matched"]).sum())
    if n_m:
        Sg.add_node("status:matched", kind="status", label="matched")
        Sg.add_edge(cnode, "status:matched", weight=n_m)
    if n_u:
        Sg.add_node("status:unmatched", kind="status", label="unmatched")
        Sg.add_edge(cnode, "status:unmatched", weight=n_u)

if Sg.number_of_nodes() >= 2:
    pos_s = nx.spring_layout(Sg, seed=3, k=2.2, iterations=100)
    if "status:matched" in Sg:
        pos_s["status:matched"] = (0.0, 1.2)
    if "status:unmatched" in Sg:
        pos_s["status:unmatched"] = (0.0, -1.2)
    fig_status = plot_nx_plotly(
        Sg, pos_s,
        title=f"Fig. 9d — Comarca ↔ status (top {TOP_N_COMARCAS_GRAPH}; tamanho∝grau)",
        color_map={"comarca": "#F58518", "status": "#54A24B"},
        legend_kinds=["comarca", "status"],
        height=520, size_scale=4.0,
    )
    show_and_save(fig_status, "fig09d_comarca_status", height=520, width=900)
    print(f"Comarca-status: nodes={Sg.number_of_nodes()} edges={Sg.number_of_edges()}")
else:
    print("Comarca-status graph skipped.")
print("Nota: grafos OAB-juiz / partes requerem join anonimizado — trabalho futuro.")


static → fig09a_bipartite_pef_comarca.png, fig09a_bipartite_pef_comarca.svg
Bipartite: nodes=31 edges=16 isolates_removed=0
Top degree nodes:


,node,degree,kind
5,com:Comarca de Barueri,2,comarca
0,pef:0014,1,pef_matched
16,com:Comarca de Diadema,1,comarca
29,pef:0625,1,pef_unmatched
28,com:Comarca de São José dos Campos,1,comarca
27,pef:0577,1,pef_unmatched
26,com:Comarca de Santo André,1,comarca
25,pef:0554,1,pef_unmatched
24,com:Comarca de Ribeirão Pires,1,comarca
23,pef:0505,1,pef_unmatched


static → fig09b_bipartite_matched_only.png, fig09b_bipartite_matched_only.svg
Small multiple (só matched): nodes=6 edges=3


static → fig09c_bipartite_unmatched_only.png, fig09c_bipartite_unmatched_only.svg
Small multiple (só unmatched (≤3 PEF/comarca)): nodes=31 edges=16


static → fig09d_comarca_status.png, fig09d_comarca_status.svg
Comarca-status: nodes=17 edges=18
Nota: grafos OAB-juiz / partes requerem join anonimizado — trabalho futuro.


## 10. Sketch de features administrativas (contagens) — matched vs unmatched

Status do sample de `debito` restrito aos PEFs do join set. **Só contagens** — sem scores, sem PII, sem rótulo “contumaz”. Analogia metodológica a outlier/risco em dados tributários (Savić et al., 2022) permanece **fora de escopo** deste toy.


In [9]:
feat = merged.copy()
feat["PEF"] = feat["PEF"].astype(str).str.strip()
feat = feat.merge(cov[["pef", "matched"]], left_on="PEF", right_on="pef", how="inner")
feat["matched"] = feat["matched"].astype(bool)

print("rows in join-set x debito merge:", len(feat))
print("matched / unmatched row counts:")
print(feat["matched"].value_counts())

for col in ["STATUS_AJUIZAMENTO_DEBITO", "SITUACAO_DEBITO", "TIPO_DEBITO"]:
    if col in feat.columns and feat[col].notna().any():
        ct = (
            feat.groupby(["matched", col], dropna=False).size()
            .rename("n").reset_index().sort_values("n", ascending=False)
        )
        print(f"\n{col} x matched (counts):")
        display(ct.head(20))
    else:
        print(f"{col}: sem overlap / coluna ausente neste sample")

summary = {
    "extracao": f"{S.ano:04d}-{S.mes:02d}",
    "window": f"{WINDOW_START.date()}…{WINDOW_END.date()}",
    "probe_n_unique_pef": probe_stats["n_unique_pef"],
    "probe_median_dt": probe_stats["median"],
    "n_ajuizamento_midwindow": int(len(aj)),
    "n_pef_join": int(n_pef),
    "sample_design": "stratified_by_year",
    "n_matched_exact": int(n_exact),
    "n_matched_normalized": int(n_norm),
    "match_rate_exact": float(rate_exact),
    "match_rate_normalized": float(rate_norm),
    "head250_exact_rate_caveat": float(rate_head),
    "digits_beyond_exact": int(n_digits_beyond),
    "n_debito": int(len(deb)) if deb is not None else 0,
    "format_anomalies_univ_a": int(n_anom_a),
    "unmatched_audit": audit,
}
summary


rows in join-set x debito merge: 250
matched / unmatched row counts:
matched
False    244
True       6
Name: count, dtype: int64
STATUS_AJUIZAMENTO_DEBITO: sem overlap / coluna ausente neste sample
SITUACAO_DEBITO: sem overlap / coluna ausente neste sample
TIPO_DEBITO: sem overlap / coluna ausente neste sample


{'extracao': '2026-03',
 'window': '2017-01-01…2020-12-31',
 'probe_n_unique_pef': 1201,
 'probe_median_dt': '2018-11-26 00:00:00',
 'n_ajuizamento_midwindow': 768,
 'n_pef_join': 250,
 'sample_design': 'stratified_by_year',
 'n_matched_exact': 6,
 'n_matched_normalized': 6,
 'match_rate_exact': 0.024,
 'match_rate_normalized': 0.024,
 'head250_exact_rate_caveat': 0.044,
 'digits_beyond_exact': 0,
 'n_debito': 300,
 'format_anomalies_univ_a': 13,
 'unmatched_audit': {'n_unmatched': 244,
  'digit_len_ne_20': 4,
  'char_len_ne_25': 4,
  'mask_build_fail': 4}}

## 11. Limitações honestas

- **Viés de amostra:** oversample da API + parquets locais ≠ desenho amostral estratificado da população `ajuizamento`. O join set é estratificado por ano *dentro do probe*, não da população estadual.
- **Incompletude do lake:** FACE é scrape (~milhões de linhas) — cobertura desigual por foro/período; match baixo com formatos alinhados aponta para esparsidade, não só para “filings recentes”.
- **Chaves:** `cd_processo` (E-SAJ) e `controle` (foro-local) **não** são PEF/CNJ; usá-los como chave infla falsos negativos.
- **`head(N)` vs estratificado:** o toy v2 reportou ~4,4% exact em `head(250)` (ordem da API). A taxa estratificada é a métrica preferida.
- **Normalização sem ganho:** digits-only / máscara CNJ não aumentaram matches neste slice (formatos já compatíveis) ⇒ $\Delta\approx 0$.
- **Cobertura descritiva ≠ linkage probabilístico** (Fellegi–Sunter): não há pesos $m$/$u$ nem classificação EM.
- **Sem PII / sem score contumaz / sem KM ainda.** Debito overlap no toy costuma ser 0 (heads independentes).


## 12. Discussão / Conclusão (sobre *estes* resultados)

### Por que o join rate é baixo?
1. **Formatos CNJ alinhados** entre dump e FACE LIMIT (quase todos 20 dígitos / 25 caracteres). Anomalias de formato no dump são raras → não explicam a taxa.
2. **Escada de match:** $\hat{c}_{\mathrm{exact}}\approx\hat{c}_{\mathrm{norm}}$ (sem lift). O gargalo não é pontuação/hífen.
3. **Caveat de ordem:** `head(250)` inflava a taxa (~4,4%) ao sobreamostrar anos com mais hits no scrape; a amostra **estratificada por ano** é mais honesta para o slice.
4. **Interpretação:** baixa cobertura dump↔lake neste toy é consistente com **esparsidade do scrape FACE** (e composição do probe), não com “só filings recentes” — a janela já é 2017–2020.

### Ângulo de artigo (ainda aberto)
Data descriptor / join study (cobertura descritiva + normalização + falhas) vs behavioral features + survival no subset matched. O toy sustenta o primeiro ângulo com números honestos; o segundo exige mais matched + features lake anonimizadas. Framing Allingham–Sandmo / Andreoni / Savić permanece **analogia**, não evidência SP.


## 13. Próximos passos (checklist paper)

1. **Protocolo full de cobertura** — export de PEFs distintos (ou amostra desenhada); join FACE `numero` com normalização; tabelas de falha por ano/comarca em escala.
2. **Features lake no matched** — classes, foros, tempos (anonimizados) para KM / riscos competitivos.
3. **Grafos OAB–juiz** — só após join anonimizado de partes.
4. **Sem** score “contumaz” a partir do dump sozinho.


## 14. Referências (framing / analogia de método)

Lista grounded: `projects/litigancia/references/referencias_grounded.md` e `docs/references/referencias_grounded.md`.

### Record linkage / cobertura (método)
- Fellegi, I. P., & Sunter, A. B. (1969). A theory for record linkage. *Journal of the American Statistical Association*, 64(328), 1183–1210. https://doi.org/10.1080/01621459.1969.10501049 — framing exact vs probabilistic; **aqui só cobertura descritiva / exact+normalizado**.
- Christen, P. (2012). *Data Matching: Concepts and Techniques for Record Linkage, Entity Resolution, and Duplicate Detection*. Springer. https://doi.org/10.1007/978-3-642-31164-2 — normalização de chaves antes do link.

### Redes / software de visualização
- Hagberg, A. A., Schult, D. A., & Swart, P. J. (2008). Exploring network structure, dynamics, and function using NetworkX. In *Proceedings of the 7th Python in Science Conference (SciPy 2008)*, 11–15. — layouts `bipartite_layout` / `spring_layout`; grafos aqui são **descritivos**.
- Asratian, A. S., Denley, T. M. R., & Häggkvist, R. (1998). *Bipartite Graphs and Their Applications*. Cambridge University Press — formulação de grafos bipartidos.

### Compliance / risco tributário (analogia apenas — não execução fiscal SP + ML)
- Allingham, M. G., & Sandmo, A. (1972). Income tax evasion: a theoretical analysis. *Journal of Public Economics*, 1(3–4), 323–338. https://doi.org/10.1016/0047-2727(72)90010-2
- Andreoni, J., Erard, B., & Feinstein, J. (1998). Tax compliance. *Journal of Economic Literature*, 36(2), 818–860. https://www.jstor.org/stable/2565123
- Savić, M., Atanasijević, J., Jakovetić, D., & Krejić, N. (2022). Tax evasion risk management using a Hybrid Unsupervised Outlier Detection method. *Expert Systems with Applications*, 193, 116409. https://doi.org/10.1016/j.eswa.2021.116409


---
**Atribuição.** Conteúdo, experimentos e conclusões: do autor. Assistência de formatação/estruturação: IA.
